## Section 1: Setup

In [1]:
import pandas as pd
import re

RAW_FILES = [
    ("../data/raw/dailynews_raw.csv",  "Daily News"),
    ("../data/raw/citizen_raw.csv",    "The Citizen"),
]
OUTPUT_FILE = "../data/processed/tz_headlines_clean.csv"

## Section 2: Combine Sources

In [2]:
frames = []
for path, source_name in RAW_FILES:
    df = pd.read_csv(path, engine='python', on_bad_lines='skip')
    df['source'] = source_name   # normalize source name at load time
    frames.append(df)
    print(f"Loaded {len(df):>5} rows — {source_name}")

df = pd.concat(frames, ignore_index=True)
print(f"\nCombined : {len(df)} rows")

Loaded  3710 rows — Daily News
Loaded  2537 rows — The Citizen

Combined : 6247 rows


## Section 3: Clean

In [3]:
original_count = len(df)

# 1. Drop duplicate URLs
df = df.drop_duplicates(subset='url')

# 2. Drop relative/unparseable dates
relative_pattern = r'yesterday|today|\d+ days? ago|hours? ago'
df = df[~df['date'].str.lower().str.contains(relative_pattern, na=True, regex=True)]

# 3. Parse dates and filter to 2022–2024
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df = df[df['date'].dt.year.between(2022, 2024)]

# 4. Drop error/placeholder headlines
error_patterns = ['error establishing', 'database connection', '404', 'page not found']
df = df[~df['headline'].str.lower().str.contains('|'.join(error_patterns), na=False)]

# 5. Drop headlines too short to be meaningful
df = df[df['headline'].str.len() >= 20]

# 6. Format date and add year_month column
df['date']       = df['date'].dt.strftime('%Y-%m-%d')
df['year_month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)

df = df.reset_index(drop=True)

print(f"Original : {original_count}")
print(f"Cleaned  : {len(df)}")
print(f"Dropped  : {original_count - len(df)}")
print(f"\nDate range : {df['date'].min()} → {df['date'].max()}")
print(f"\nBy source :\n{df['source'].value_counts().to_string()}")
print(f"\nBy year   :\n{df['year_month'].str[:4].value_counts().sort_index().to_string()}")

Original : 6247
Cleaned  : 6235
Dropped  : 12

Date range : 2022-01-01 → 2024-12-31

By source :
source
Daily News     3699
The Citizen    2536

By year   :
year_month
2022    1665
2023    2105
2024    2465


In [4]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

Saved 6235 rows to ../data/processed/tz_headlines_clean.csv


## Section 4: Save